In [1]:
import osmnx as ox
import math
import pandas as pd
import numpy
from osgeo import gdal
# OSMnx konfigurieren
ox.config(use_cache=True, log_console=True)
ox.settings.useful_tags_way = ['segregated','class:bicycle','cycleway:buffer','bus','bridge', 'tunnel', 'oneway', 'lanes','foot', 'ref', 'name',
                    'highway', 'maxspeed', 'access', 'area','landuse',
                    'width','cycle_network', 'est_width', 'junction', 'surface', 'bicycle', 'traffic_sign','oneway:bicycle'
                    'cycle_barrier', 'cycleway','cycleway:both:lane', 'cycleway:both','smoothness','parking','parking:both','parking:lane:both','parking:lane:right','parking:lane:left',
                    'cycleway:right','cycleway:right:lane','junction','level','class:bicycle', 'tracktype', 
                    'cycleway:left', 'cycleway:left:lane','bicycle:conditional','oneway:bicycle','cycleway:surface', 'bicycle_road',
                    'cycleway:width','cycleway:lane','hgv','cycleway:left:segregated','cycleway:right:segregated']

# CSV-Datei einlesen
file_path = 'random_coordinates.csv'
df = pd.read_csv(file_path)

# Ergebnis DataFrame vorbereiten
results = pd.DataFrame()

for index, row in df.iterrows():

    latitude, longitude = row['lat'], row['lon']

    meters_per_lat = 111320
    meters_per_lon = 40075000 * math.cos(math.radians(latitude)) / 360
    delta_lat = 5000/ meters_per_lat
    delta_lon = 5000/ meters_per_lon
    north = latitude + delta_lat
    south = latitude
    east = longitude + delta_lon
    west = longitude
    bbox = (west, south, east, north)

    G6 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)

    # Entfernen von "service" Kanten
    service_edges = [(u, v, k) for u, v, k, d in G6.edges(keys=True, data=True) if d.get('highway') == 'service']
    G6.remove_edges_from(service_edges)

    def add_elevation_and_slope(G6, raster_path):
        G6 = ox.elevation.add_node_elevations_raster(G6, raster_path)
        G6 = ox.elevation.add_edge_grades(G6, add_absolute=True)
        return G6
    #Rasterdatei mit Höhenlinien werden eingespielt
    raster_path = "srtm_germany_dtm.tif"
    #Funktion wird auf diese Rasterdatei angewandt
    G6 = add_elevation_and_slope(G6, raster_path)

    ##Steigung Bewertung

    #Umwandlung in GeoDataFrame
    edges = ox.graph_to_gdfs(G6, nodes=False)
    #Negative Grade werden als 0 gewertet, bevor die Steigung sich aufhebt
    #edges['grade'] = edges['grade'].apply(lambda x: max(x, 0))
    #Negative Steigung wird entfernt und nicht bewertet, da diese die eigenttliche Steigung beeinflussen
    edges = edges[edges['grade'] >= 0]
    #Umwandlung in Prozent 
    edges['grade'] = edges['grade'] * 100
    #Funktion zur Bewertung der Steigung (grade)
      # Zuordnung der Steigungsklassen
    def grade_score(grade):
        if grade > 15:
            return 1
        elif 10 < grade:
            return 2
        elif 5 < grade :
            return 4
        elif 2 < grade:
            return 6
        elif 0.5 < grade:
            return 8
        else:
            return 10
       
        
    # Anwenden der Bewertungsfunktionen auf die Daten  
    edges['grade_score'] = edges['grade'].apply(grade_score)
    # Berechnen der gewichteten Scores (Steigung)
    edges['weighted_grade_score'] = edges['grade_score'] * edges['length']
    # Berechnen des Gesamtwerts der gewichteten Scores
    total_weighted_grade_score = edges['weighted_grade_score'].sum()
    # Berechnen der Gesamtlänge aller Kanten
    total_length = edges['length'].sum()
    # Berechnen des durchschnittlichen gewichteten 'grade_score'
    s_score = total_weighted_grade_score / total_length

    print(f'Gewichteter Slope_Score: {s_score}')


    #Bicycling Infrastructure

    non_cyc = []

    for u, v, k, d in G6.edges(keys=True, data=True):
        if d.get('bicycle') == 'separate' or d.get('cycleway') == 'separate' \
        or d.get('cycleway:right') == 'separate' or d.get('cycleway:left') == 'separate' \
        or d.get('cycleway:both') == 'separate':
            non_cyc.append((u, v, k))
        if d.get('bicycle') in ['designated', 'use_sidepath']:
            continue
        elif d.get('highway') in ['cycleway'] :
            continue    
        elif d.get('cycleway') in ['track','lane', 'opposite_track']:
            continue
        elif d.get('cycleway:right') in ['track','lane', 'opposite_track']:
            continue
        elif d.get('cycleway:left') in ['track','lane', 'opposite_track']:
            continue
        elif d.get('cycleway:both') in ['track','lane', 'opposite_track']:
            continue   
        elif d.get('highway') == 'path' and d.get('bicycle') == 'yes':
            continue
        non_cyc.append((u, v, k))
    G6.remove_edges_from(non_cyc)
    G6 = ox.utils_graph.remove_isolated_nodes(G6)
    stats2 = ox.stats.basic_stats(G6)
    streetlength = stats2['street_length_total']
    # Skalierung der Bewertungsbereiche mit Rundung auf ganze Zahlen
    scale_factor = 127.324
    scaled_ranges = {
        1: (round(0 * scale_factor), round(0 * scale_factor)),                   # 0-0 m (hochskaliert)
        2: (round(1), round(251 * scale_factor)),                                # 1-31957 m (hochskaliert)
        3: (round(251 * scale_factor + 1), round(430 * scale_factor)),           # 31958-54753 m (hochskaliert)
        4: (round(430 * scale_factor + 1), round(545 * scale_factor)),           # 54754-69371 m (hochskaliert)
        5: (round(545 * scale_factor + 1), round(702 * scale_factor)),           # 69372-89356 m (hochskaliert)
        6: (round(702 * scale_factor + 1), round(874 * scale_factor)),           # 89357-111314 m (hochskaliert)
        7: (round(874 * scale_factor + 1), round(1063 * scale_factor)),          # 111315-135355 m (hochskaliert)
        8: (round(1063 * scale_factor + 1), round(1314 * scale_factor)),         # 135356-167219 m (hochskaliert)
        9: (round(1314 * scale_factor + 1), round(1645 * scale_factor)),         # 167220-209380 m (hochskaliert)
        10: (round(1645 * scale_factor + 1), round(3353 * scale_factor))         # 209381-426767 m (hochskaliert)
    }

    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich

    for key, (low, high) in scaled_ranges.items():
        if low <= streetlength <= high:
            R_score = key
            break
    print("Länge der Fahrradroute:", streetlength)
    print("Bewertungspunkt:", R_score)


    # Erstellen des Graphen G7
    G7 = ox.graph_from_bbox(north, south, east, west, network_type='bike', simplify=True, retain_all=True, truncate_by_edge=True)
    # Entfernen von "service" Kanten
    service_edges = [(u, v, k) for u, v, k, d in G7.edges(keys=True, data=True) if d.get('highway') == 'service']
    G7.remove_edges_from(service_edges)


    #Bike Separation
    scale_factor = 127.324
    min_sep = scale_factor * 10
    def evaluate_bicycle_route_separation(G7):
        # Gesamtlänge der getrennten Fahrradwege berechnen
        separated_cycleways_length = sum(
            d['length'] for u, v, k, d in G7.edges(keys=True, data=True)
            if d.get('highway') == 'cycleway'#or d.get('bicycle') == 'designated' 
        )
        print(f"Bicycle Route Separation Length: {separated_cycleways_length}")
        # Wenn die Länge der getrennten Fahrradwege mindestens min_sep ist, geben Sie eine hohe Bewertung zurück, sonst eine niedrige
        return 10 if separated_cycleways_length >= min_sep else 1

    # Führen Sie die Bewertung für den Graphen G6 durch
    S_score = evaluate_bicycle_route_separation(G7)

    print(f"Bicycle Route Separation Score: {S_score}")



    # Main Roads

    G8 = ox.graph_from_bbox(north, south, east, west, network_type='drive', simplify=True, retain_all=True, truncate_by_edge=True)
   
    # Entfernen von "service" Kanten
    service_edges = [(u, v, k) for u, v, k, d in G8.edges(keys=True, data=True) if d.get('highway') == 'service']
    

    # Liste der zu entfernenden Kanten
    edges_to_remove = []
    for u, v, key, data in G8.edges(keys=True, data=True):
        if data.get('bicycle') == 'separate' or \
        data.get('cycleway') == 'separate' or \
        data.get('cycleway:right') == 'separate' or \
        data.get('cycleway:left') == 'separate' or \
        data.get('cycleway:both') == 'separate' or \
        data.get('bicycle') in ['designated', 'yes','use_sidepath'] or \
        data.get('highway') in ['path','residential', 'cycleway', 'pedestrian', 'track', 'raceway', 'living_street'] or \
        data.get('cycleway') in ['track', 'lane', 'shared_lane'] or \
        data.get('cycleway:right') in ['track', 'lane', 'shared_lane'] or \
        data.get('cycleway:left') in ['track', 'lane', 'shared_lane'] or \
        data.get('cycleway:both') in ['track', 'lane', 'shared_lane']:
            edges_to_remove.append((u, v, key))

    # Entfernen der Kanten aus dem Graphen
    G8.remove_edges_from(edges_to_remove + service_edges)

    # Berechnen der Gesamtlänge der Hauptstraßen
    stats2 = ox.stats.basic_stats(G8)
    length = stats2['street_length_total']
    


    # Konvertieren des Graphen zu GeoDataFrame
    edges7 = ox.graph_to_gdfs(G8, nodes=False)

    scaled_ranges = {
        1: (round(1101 * scale_factor +1), round(3565 * scale_factor)), 
        2: (round(883 * scale_factor +1), round(1101 * scale_factor)),                                # 1-31957 m (hochskaliert)
        3: (round(726 * scale_factor + 1), round(883 * scale_factor)),           # 31958-54753 m (hochskaliert)
        4: (round(585 * scale_factor + 1), round(726 * scale_factor)),           # 54754-69371 m (hochskaliert)
        5: (round(491 * scale_factor + 1), round(585 * scale_factor)),           # 69372-89356 m (hochskaliert)
        6: (round(405 * scale_factor + 1), round(491 * scale_factor)),           # 89357-111314 m (hochskaliert)
        7: (round(288 * scale_factor + 1), round(405 * scale_factor)),          # 111315-135355 m (hochskaliert)
        8: (round(160 * scale_factor + 1), round(288 * scale_factor)),         # 135356-167219 m (hochskaliert)
        9: (round(1), round(160 * scale_factor)),         # 167220-209380 m (hochskaliert)
        10: (round(0), round(0))         # 209381-426767 m (hochskaliert)
    }

    # Ermittlung der Bewertung
    # Standardwert für Werte über dem höchsten Bereich

    for key, (low, high) in scaled_ranges.items():
        if low <= length <= high:
            M_score = key
            break
    print(f"Main Roads Length: {length}")
    print(f"Main Roads Length Score: {M_score}")


    ##Anteil Grünfläche


    import osmnx as ox
    import geopandas as gpd
    from shapely.geometry import box
    import math

    # Funktion zur Bewertung des Grünflächenanteils
    
    def bewerte_gruenflaechen(anteil):
        if anteil == 0:
            return 0
        elif 0 < anteil <= (scale_factor * 9955):
            return 1
        elif (9955* scale_factor)< anteil <= (18241 * scale_factor):
            return 2
        elif (18241 * scale_factor) < anteil <= (scale_factor * 29039):
            return 3
        elif (scale_factor * 29039) < anteil <= (scale_factor * 43163):
            return 4
        elif (scale_factor * 43163) < anteil <= (scale_factor * 60693):
            return 5
        elif (scale_factor * 60693) < anteil <= (scale_factor * 81508):
            return 6
        elif (scale_factor * 81508) < anteil <= (scale_factor * 108159):
            return 7
        elif (scale_factor * 108159) < anteil <= (scale_factor * 139712):
            return 8
        elif (scale_factor * 139712) < anteil <= (scale_factor * 161999):
            return 9
        elif anteil > (scale_factor * 161999):
            return 10
        else:
            return "Anteil außerhalb des definierten Bereichs"

    tags = {'leisure': 'park', 'landuse': ['grass','forest','meadow','recreation_ground'], 'natural': ['water','scrub', 'wetland', 'coastline', 'sand']}

    # Extrahieren von Grünflächen innerhalb der Bounding Box mit OSMnx
    green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)

    # Erstellen eines GeoDataFrame für die Grünflächen
    green_spaces_gdf = gpd.GeoDataFrame(green_spaces, geometry='geometry', crs="EPSG:4326")

    # Transformieren in die flächentreue Projektion EPSG:3035 für die Flächenberechnung
    green_spaces_gdf = green_spaces_gdf.to_crs("EPSG:3035")

    # Berechnen der Gesamtfläche der Grünflächen in Quadratkilometern
    total_green_area = green_spaces_gdf.area.sum()



    # Anwenden des Bewertungssystems
    g_score = bewerte_gruenflaechen(total_green_area)

    print(f"Grünfläche in m²: {total_green_area}")
    print(f"Bewertung Grünfläche: {g_score}")


    #Endbewertung
    Note = (s_score + S_score + R_score+ M_score + g_score)/5
    print(f"Endbewertung: {Note}")


    # Hinzufügen der Ergebnisse zur Ergebnis-DataFrame
    results.at[index, 'latitude'] = latitude
    results.at[index, 'longitude'] = longitude

    results.at[index, 'Main_Roads_Bewertung'] = M_score
    results.at[index, 'Route_Length_Bewertung'] = R_score
    results.at[index, 'Route_Separation_Bewertung'] = S_score
    results.at[index, 'Steigung_Bewertung'] = s_score
    results.at[index, 'Grünfläche_Bewertung'] = g_score

    results.at[index, 'Endbewertung'] = Note

# Ergebnis als CSV speichern
results.to_csv('Bewertungsergebnisse_Krenn.csv', index=False)

C:\Users\kevdr\AppData\Local\Temp\ipykernel_6992\2655728729.py:7: UserWarning: The `utils.config` function is deprecated and will be removed in a future release. Instead, use the `settings` module directly to configure a global setting's value. For example, `ox.settings.log_console=True`.
  ox.config(use_cache=True, log_console=True)


Gewichteter Slope_Score: 7.359716151872211
Länge der Fahrradroute: 65730.79299999996
Bewertungspunkt: 4
Bicycle Route Separation Length: 2259.414
Bicycle Route Separation Score: 10
Main Roads Length: 83065.75299999994
Main Roads Length Score: 4


C:\Users\kevdr\AppData\Local\Temp\ipykernel_6992\2655728729.py:273: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in m²: 8066039.94414345
Bewertung Grünfläche: 6
Endbewertung: 6.2719432303744425
Gewichteter Slope_Score: 8.484997212216243
Länge der Fahrradroute: 144484.20499999975
Bewertungspunkt: 8
Bicycle Route Separation Length: 11533.713999999998
Bicycle Route Separation Score: 10
Main Roads Length: 67572.69099999999
Main Roads Length Score: 5


C:\Users\kevdr\AppData\Local\Temp\ipykernel_6992\2655728729.py:273: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


...

Grünfläche in m²: 28617248.912809905
Bewertung Grünfläche: 10
Endbewertung: 5.383045145893243
Gewichteter Slope_Score: 8.50401652656731
Länge der Fahrradroute: 120589.3460000002
Bewertungspunkt: 7
Bicycle Route Separation Length: 6787.915000000002
Bicycle Route Separation Score: 10
Main Roads Length: 49872.37799999997
Main Roads Length Score: 7


C:\Users\kevdr\AppData\Local\Temp\ipykernel_6992\2655728729.py:273: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in m²: 7224156.363829805
Bewertung Grünfläche: 5
Endbewertung: 7.500803305313463
Gewichteter Slope_Score: 8.08903037054941
Länge der Fahrradroute: 63385.34800000002
Bewertungspunkt: 4
Bicycle Route Separation Length: 12296.211000000008
Bicycle Route Separation Score: 10
Main Roads Length: 64093.64400000002
Main Roads Length Score: 5


C:\Users\kevdr\AppData\Local\Temp\ipykernel_6992\2655728729.py:273: UserWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in a future release.
  green_spaces = ox.geometries_from_bbox(north, south, east, west, tags=tags)


Grünfläche in m²: 8871337.489684977
Bewertung Grünfläche: 6
Endbewertung: 6.617806074109882


In [11]:
total_green_area

36173568.29411008

Main Roads

In [ ]:
import osmnx as ox

# Erstellen des Graphen G7
G7 = ox.graph_from_bbox(north, south, east, west, network_type='drive', simplify=True, retain_all=True, truncate_by_edge=True)

# Liste der zu entfernenden Kanten
edges_to_remove = []
for u, v, key, data in G7.edges(keys=True, data=True):
    if data.get('bicycle') == 'separate' or \
       data.get('cycleway') == 'separate' or \
       data.get('cycleway:right') == 'separate' or \
       data.get('cycleway:left') == 'separate' or \
       data.get('cycleway:both') == 'separate' or \
       data.get('bicycle') in ['designated', 'use_sidepath'] or \
       data.get('highway') in ['residential', 'cycleway', 'pedestrian', 'track', 'raceway', 'living_street']  or \
       data.get('cycleway') in ['track', 'lane', 'shared_lane'] or \
       data.get('cycleway:right') in ['track', 'lane', 'shared_lane'] or \
       data.get('cycleway:left') in ['track', 'lane', 'shared_lane'] or \
       data.get('cycleway:both') in ['track', 'lane', 'shared_lane']:
        edges_to_remove.append((u, v, key))

# Entfernen der Kanten aus dem Graphen
G7.remove_edges_from(edges_to_remove)

# Berechnen der Gesamtlänge der Hauptstraßen
length = ox.stats.edge_length_total(G7)
print(length)

# Konvertieren des Graphen zu GeoDataFrame
edges7 = ox.graph_to_gdfs(G7, nodes=False)
fig,ax = ox.plot_graph(G7)

scaled_ranges = {
    1: (round(1101 * scale_factor +1), round(3565 * scale_factor)), 
    2: (round(883 * scale_factor +1), round(1101 * scale_factor)),                                # 1-31957 m (hochskaliert)
    3: (round(726 * scale_factor + 1), round(883 * scale_factor)),           # 31958-54753 m (hochskaliert)
    4: (round(585 * scale_factor + 1), round(726 * scale_factor)),           # 54754-69371 m (hochskaliert)
    5: (round(491 * scale_factor + 1), round(585 * scale_factor)),           # 69372-89356 m (hochskaliert)
    6: (round(405 * scale_factor + 1), round(491 * scale_factor)),           # 89357-111314 m (hochskaliert)
    7: (round(288 * scale_factor + 1), round(405 * scale_factor)),          # 111315-135355 m (hochskaliert)
    8: (round(160 * scale_factor + 1), round(288 * scale_factor)),         # 135356-167219 m (hochskaliert)
    9: (round(1), round(160 * scale_factor)),         # 167220-209380 m (hochskaliert)
    10: (round(0), round(0))         # 209381-426767 m (hochskaliert)
}

# Ermittlung der Bewertung
# Standardwert für Werte über dem höchsten Bereich

for key, (low, high) in scaled_ranges.items():
    if low <= streetlength <= high:
        M_score = key
        break

print(f"Main Roads Length Score: {M_score}")

